In [1]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re

In [2]:
with open('321615.log', 'r') as f:
    data = json.load(f)
activitiesLog = data["activitiesLog"]

In [3]:
df_act = pd.read_csv(StringIO(data["activitiesLog"]), sep=";")
df_act["mid_price"] = df_act["mid_price"].replace(0, np.nan)
df_act.dropna(subset=["mid_price"], inplace=True)

In [4]:
df_trades = pd.DataFrame(data["tradeHistory"])
df_trades["side"] = np.where(
    df_trades["buyer"] == "SUBMISSION", "BUY",
    np.where(df_trades["seller"] == "SUBMISSION", "SELL", "MARKET")
)
df_trades["signed_qty"] = np.where(
    df_trades["side"] == "SELL", -df_trades["quantity"], df_trades["quantity"]
)

In [5]:
def parse_positions_from_logs(raw_logs):
    """Extrait les positions à chaque timestamp."""
    rows = []
    for entry in raw_logs:
        ts = entry["timestamp"]
        ll = entry.get("lambdaLog", "")
        m = re.search(r"'POSITIONS':\s*(\{[^}]*\})", ll)
        if m:
            pos_str = m.group(1).replace("'", '"')
            try:
                positions = json.loads(pos_str)
                for sym, qty in positions.items():
                    rows.append({"timestamp": ts, "symbol": sym, "position": qty})
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows)

In [6]:
def parse_orders_from_logs(raw_logs):
    """Extrait les ordres envoyés à chaque timestamp depuis lambdaLog."""
    rows = []
    for entry in raw_logs:
        ts = entry["timestamp"]
        ll = entry.get("lambdaLog", "")

        for m in re.finditer(r"'BUYO':\s*\{'p':\s*(\d+),\s*'s':\s*'(\w+)',\s*'v':\s*(-?\d+)\}", ll):
            rows.append({
                "timestamp": ts,
                "type": "BUY",
                "symbol": m.group(2),
                "price": int(m.group(1)),
                "volume": int(m.group(3)),
            })
        # Extraire les ordres SELL
        for m in re.finditer(r"'SELLO':\s*\{'p':\s*(\d+),\s*'s':\s*'(\w+)',\s*'v':\s*(-?\d+)\}", ll):
            rows.append({
                "timestamp": ts,
                "type": "SELL",
                "symbol": m.group(2),
                "price": int(m.group(1)),
                "volume": int(m.group(3)),
            })
    return pd.DataFrame(rows)

In [7]:
def parse_fair_values(raw_logs):
    """Extrait les fair values calculées par ton algo (si présentes dans le log)."""
    rows = []
    for entry in raw_logs:
        ts = entry["timestamp"]
        ll = entry.get("lambdaLog", "")
        for m in re.finditer(r"'FV':\s*([\d.]+)", ll):
            preceding = ll[:m.start()]
            sym_match = list(re.finditer(r"'(\w+)':\s*\{", preceding))
            if sym_match:
                sym = sym_match[-1].group(1)
                rows.append({"timestamp": ts, "symbol": sym, "fair_value": float(m.group(1))})
    return pd.DataFrame(rows)

In [8]:
raw_logs = data["logs"]

In [9]:
df_orders = parse_orders_from_logs(raw_logs)
df_positions = parse_positions_from_logs(raw_logs)
df_fv = parse_fair_values(raw_logs)

my_trades = df_trades[df_trades["side"].isin(["BUY", "SELL"])].copy()


In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SYM = "ASH_COATED_OSMIUM"

act = df_act[df_act["product"] == SYM].copy()
orders_sym = df_orders[df_orders["symbol"] == SYM].copy()
trades_sym = my_trades[my_trades["symbol"] == SYM].copy()

filled_keys = set(zip(trades_sym["timestamp"], trades_sym["price"]))
orders_sym["filled"] = orders_sym.apply(
    lambda r: (r["timestamp"], r["price"]) in filled_keys, axis=1
)

orders_filled = orders_sym[orders_sym["filled"]]
orders_unfilled = orders_sym[~orders_sym["filled"]]

buy_filled = orders_filled[orders_filled["type"] == "BUY"]
sell_filled = orders_filled[orders_filled["type"] == "SELL"]
buy_unfilled = orders_unfilled[orders_unfilled["type"] == "BUY"]
sell_unfilled = orders_unfilled[orders_unfilled["type"] == "SELL"]

buy_filled = buy_filled.merge(
    trades_sym[trades_sym["side"] == "BUY"][["timestamp", "price", "quantity"]],
    on=["timestamp", "price"], how="left"
)
sell_filled = sell_filled.merge(
    trades_sym[trades_sym["side"] == "SELL"][["timestamp", "price", "quantity"]],
    on=["timestamp", "price"], how="left"
)

pos_sym = df_positions[df_positions["symbol"] == SYM].copy() if len(df_positions) > 0 else pd.DataFrame()

if len(pos_sym) == 0:
    trades_for_pos = trades_sym.sort_values("timestamp").copy()
    trades_for_pos["signed_qty"] = np.where(
        trades_for_pos["side"] == "BUY",
        trades_for_pos["quantity"],
        -trades_for_pos["quantity"]
    )
    trades_for_pos["position"] = trades_for_pos["signed_qty"].cumsum()
    pos_sym = trades_for_pos[["timestamp", "position"]]

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.60, 0.20, 0.20],
    subplot_titles=[f"{SYM} — Order Book + Ordres", "Position", "PnL"],
    vertical_spacing=0.06,
)
book_styles = {
    "bid": {
        1: {"color": "rgba(0,180,0,0.9)",  "width": 1.4},   # best bid — vert vif
        2: {"color": "rgba(0,140,0,0.55)", "width": 1.0},   # vert moyen
        3: {"color": "rgba(0,100,0,0.3)",  "width": 0.7},   # vert pâle
    },
    "ask": {
        1: {"color": "rgba(220,0,0,0.9)",  "width": 1.4},   # best ask — rouge vif
        2: {"color": "rgba(180,0,0,0.55)", "width": 1.0},   # rouge moyen
        3: {"color": "rgba(140,0,0,0.3)",  "width": 0.7},   # rouge pâle
    },
}

for level in [3, 2, 1]:  # dessiner les profonds d'abord
    bid_col = f"bid_price_{level}"
    ask_col = f"ask_price_{level}"

    mask_bid = act[bid_col].notna() & (act[bid_col] > 0)
    mask_ask = act[ask_col].notna() & (act[ask_col] > 0)

    bid_series = act[bid_col].copy()
    bid_series[~mask_bid] = np.nan
    ask_series = act[ask_col].copy()
    ask_series[~mask_ask] = np.nan

    fig.add_trace(go.Scattergl(
        x=act["timestamp"],
        y=bid_series,
        mode="lines",
        line=dict(
            color=book_styles["bid"][level]["color"],
            width=book_styles["bid"][level]["width"],
        ),
        name=f"Bid {level}",
        legendgroup="book_bid",
        showlegend=True,
        connectgaps=False,
    ), row=1, col=1)

    fig.add_trace(go.Scattergl(
        x=act["timestamp"],
        y=ask_series,
        mode="lines",
        line=dict(
            color=book_styles["ask"][level]["color"],
            width=book_styles["ask"][level]["width"],
        ),
        name=f"Ask {level}",
        legendgroup="book_ask",
        showlegend=True,
        connectgaps=False,
    ), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=act["timestamp"], y=act["mid_price"],
    mode="lines", name="Mid Price",
    line=dict(color="white", width=1.2, dash="dot"),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=buy_unfilled["timestamp"], y=buy_unfilled["price"],
    mode="markers", name="Buy ordre (unfilled)",
    marker=dict(size=3, color="rgba(0,200,0,0.15)", symbol="circle"),
    hovertemplate="BUY unfilled<br>t=%{x}<br>price=%{y}<br>vol=%{text}<extra></extra>",
    text=buy_unfilled["volume"].astype(str),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=sell_unfilled["timestamp"], y=sell_unfilled["price"],
    mode="markers", name="Sell ordre (unfilled)",
    marker=dict(size=3, color="rgba(200,0,0,0.15)", symbol="circle"),
    hovertemplate="SELL unfilled<br>t=%{x}<br>price=%{y}<br>vol=%{text}<extra></extra>",
    text=sell_unfilled["volume"].abs().astype(str),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=buy_filled["timestamp"], y=buy_filled["price"],
    mode="markers", name="BUY FILLED ✓",
    marker=dict(
        size=buy_filled["quantity"].clip(upper=12) + 4,
        color="lime", line=dict(width=1, color="darkgreen"),
        symbol="triangle-up",
    ),
    hovertemplate="BUY FILLED<br>t=%{x}<br>price=%{y}<br>qty=%{text}<extra></extra>",
    text=buy_filled["quantity"].astype(str),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=sell_filled["timestamp"], y=sell_filled["price"],
    mode="markers", name="SELL FILLED ✓",
    marker=dict(
        size=sell_filled["quantity"].clip(upper=12) + 4,
        color="red", line=dict(width=1, color="darkred"),
        symbol="triangle-down",
    ),
    hovertemplate="SELL FILLED<br>t=%{x}<br>price=%{y}<br>qty=%{text}<extra></extra>",
    text=sell_filled["quantity"].astype(str),
), row=1, col=1)

if len(df_fv) > 0:
    fv_sym = df_fv[df_fv["symbol"] == SYM]
    if len(fv_sym) > 0:
        fig.add_trace(go.Scattergl(
            x=fv_sym["timestamp"], y=fv_sym["fair_value"],
            mode="lines", name="Fair Value (algo)",
            line=dict(color="dodgerblue", width=1.5, dash="dash"),
        ), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=pos_sym["timestamp"], y=pos_sym["position"],
    mode="lines", name="Position",
    line=dict(color="orange", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(255,165,0,0.15)",
), row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

fig.add_trace(go.Scattergl(
    x=act["timestamp"], y=act["profit_and_loss"],
    mode="lines", name="PnL",
    line=dict(color="purple", width=1.5),
), row=3, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

fig.update_layout(
    height=900,
    width=1400,
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
        font=dict(size=10),
    ),
    title_text=f"Analyse {SYM} — Book + Ordres + Position",
    template="plotly_dark",
    hovermode="closest",
)

fig.update_xaxes(title_text="Timestamp", row=3, col=1)
fig.update_yaxes(title_text="Prix", row=1, col=1)
fig.update_yaxes(title_text="Pos", row=2, col=1)
fig.update_yaxes(title_text="PnL", row=3, col=1)

fig.show()

In [11]:
SYM = "INTARIAN_PEPPER_ROOT"

act = df_act[df_act["product"] == SYM].copy()
orders_sym = df_orders[df_orders["symbol"] == SYM].copy()
trades_sym = my_trades[my_trades["symbol"] == SYM].copy()

filled_keys = set(zip(trades_sym["timestamp"], trades_sym["price"]))
orders_sym["filled"] = orders_sym.apply(
    lambda r: (r["timestamp"], r["price"]) in filled_keys, axis=1
)

orders_filled = orders_sym[orders_sym["filled"]]
orders_unfilled = orders_sym[~orders_sym["filled"]]

buy_filled = orders_filled[orders_filled["type"] == "BUY"]
sell_filled = orders_filled[orders_filled["type"] == "SELL"]
buy_unfilled = orders_unfilled[orders_unfilled["type"] == "BUY"]
sell_unfilled = orders_unfilled[orders_unfilled["type"] == "SELL"]

buy_filled = buy_filled.merge(
    trades_sym[trades_sym["side"] == "BUY"][["timestamp", "price", "quantity"]],
    on=["timestamp", "price"], how="left"
)
buy_filled["quantity"] = buy_filled["quantity"].fillna(0)

sell_filled = sell_filled.merge(
    trades_sym[trades_sym["side"] == "SELL"][["timestamp", "price", "quantity"]],
    on=["timestamp", "price"], how="left"
)
sell_filled["quantity"] = sell_filled["quantity"].fillna(0)

pos_sym = df_positions[df_positions["symbol"] == SYM].copy() if len(df_positions) > 0 else pd.DataFrame()

if len(pos_sym) == 0:
    trades_for_pos = trades_sym.sort_values("timestamp").copy()
    trades_for_pos["signed_qty"] = np.where(
        trades_for_pos["side"] == "BUY",
        trades_for_pos["quantity"],
        -trades_for_pos["quantity"]
    )
    trades_for_pos["position"] = trades_for_pos["signed_qty"].cumsum()
    pos_sym = trades_for_pos[["timestamp", "position"]]

# --- Figure 3 subplots ---
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.60, 0.20, 0.20],
    subplot_titles=[f"{SYM} — Order Book + Ordres", "Position", "PnL"],
    vertical_spacing=0.06,
)

# ========================
#  ORDER BOOK — lignes avec dégradé de couleur
# ========================
book_styles = {
    "bid": {
        1: {"color": "rgba(0,180,0,0.9)",  "width": 1.4},
        2: {"color": "rgba(0,140,0,0.55)", "width": 1.0},
        3: {"color": "rgba(0,100,0,0.3)",  "width": 0.7},
    },
    "ask": {
        1: {"color": "rgba(220,0,0,0.9)",  "width": 1.4},
        2: {"color": "rgba(180,0,0,0.55)", "width": 1.0},
        3: {"color": "rgba(140,0,0,0.3)",  "width": 0.7},
    },
}

for level in [3, 2, 1]:
    bid_col = f"bid_price_{level}"
    ask_col = f"ask_price_{level}"

    mask_bid = act[bid_col].notna() & (act[bid_col] > 0)
    mask_ask = act[ask_col].notna() & (act[ask_col] > 0)

    bid_series = act[bid_col].copy()
    bid_series[~mask_bid] = np.nan
    ask_series = act[ask_col].copy()
    ask_series[~mask_ask] = np.nan

    fig.add_trace(go.Scattergl(
        x=act["timestamp"], y=bid_series,
        mode="lines",
        line=dict(color=book_styles["bid"][level]["color"], width=book_styles["bid"][level]["width"]),
        name=f"Bid {level}", legendgroup="book_bid", showlegend=True, connectgaps=False,
    ), row=1, col=1)

    fig.add_trace(go.Scattergl(
        x=act["timestamp"], y=ask_series,
        mode="lines",
        line=dict(color=book_styles["ask"][level]["color"], width=book_styles["ask"][level]["width"]),
        name=f"Ask {level}", legendgroup="book_ask", showlegend=True, connectgaps=False,
    ), row=1, col=1)

# ========================
#  MID PRICE
# ========================
fig.add_trace(go.Scattergl(
    x=act["timestamp"], y=act["mid_price"],
    mode="lines", name="Mid Price",
    line=dict(color="white", width=1.2, dash="dot"),
), row=1, col=1)

# ========================
#  ORDRES UNFILLED
# ========================
fig.add_trace(go.Scattergl(
    x=buy_unfilled["timestamp"], y=buy_unfilled["price"],
    mode="markers", name="Buy ordre (unfilled)",
    marker=dict(size=3, color="rgba(0,200,0,0.15)", symbol="circle"),
    hovertemplate="BUY unfilled<br>t=%{x}<br>price=%{y}<br>vol=%{text}<extra></extra>",
    text=buy_unfilled["volume"].astype(str),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=sell_unfilled["timestamp"], y=sell_unfilled["price"],
    mode="markers", name="Sell ordre (unfilled)",
    marker=dict(size=3, color="rgba(200,0,0,0.15)", symbol="circle"),
    hovertemplate="SELL unfilled<br>t=%{x}<br>price=%{y}<br>vol=%{text}<extra></extra>",
    text=sell_unfilled["volume"].abs().astype(str),
), row=1, col=1)

# ========================
#  ORDRES FILLED
# ========================
fig.add_trace(go.Scattergl(
    x=buy_filled["timestamp"], y=buy_filled["price"],
    mode="markers", name="BUY FILLED ✓",
    marker=dict(
        size=buy_filled["quantity"].clip(upper=12) + 4,
        color="lime", line=dict(width=1, color="darkgreen"),
        symbol="triangle-up",
    ),
    hovertemplate="BUY FILLED<br>t=%{x}<br>price=%{y}<br>qty=%{text}<extra></extra>",
    text=buy_filled["quantity"].astype(str),
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=sell_filled["timestamp"], y=sell_filled["price"],
    mode="markers", name="SELL FILLED ✓",
    marker=dict(
        size=sell_filled["quantity"].clip(upper=12) + 4,
        color="red", line=dict(width=1, color="darkred"),
        symbol="triangle-down",
    ),
    hovertemplate="SELL FILLED<br>t=%{x}<br>price=%{y}<br>qty=%{text}<extra></extra>",
    text=sell_filled["quantity"].astype(str),
), row=1, col=1)

# ========================
#  FAIR VALUE
# ========================
if len(df_fv) > 0:
    fv_sym = df_fv[df_fv["symbol"] == SYM]
    if len(fv_sym) > 0:
        fig.add_trace(go.Scattergl(
            x=fv_sym["timestamp"], y=fv_sym["fair_value"],
            mode="lines", name="Fair Value (algo)",
            line=dict(color="dodgerblue", width=1.5, dash="dash"),
        ), row=1, col=1)

# ========================
#  POSITION (row 2)
# ========================
fig.add_trace(go.Scattergl(
    x=pos_sym["timestamp"], y=pos_sym["position"],
    mode="lines", name="Position",
    line=dict(color="orange", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(255,165,0,0.15)",
), row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

# ========================
#  PNL (row 3)
# ========================
fig.add_trace(go.Scattergl(
    x=act["timestamp"], y=act["profit_and_loss"],
    mode="lines", name="PnL",
    line=dict(color="purple", width=1.5),
), row=3, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

# ========================
#  LAYOUT
# ========================
fig.update_layout(
    height=900,
    width=1400,
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
        font=dict(size=10),
    ),
    title_text=f"Analyse {SYM} — Book + Ordres + Position",
    template="plotly_dark",
    hovermode="closest",
)

fig.update_xaxes(title_text="Timestamp", row=3, col=1)
fig.update_yaxes(title_text="Prix", row=1, col=1)
fig.update_yaxes(title_text="Pos", row=2, col=1)
fig.update_yaxes(title_text="PnL", row=3, col=1)

fig.show()
